# 🔬 Notebook 2: Guided Acoustic Calibration Lab ($k(f)$ Extraction)
Welcome to the **Acoustic Calibration Protocol Suite** (`v1.1.0`).

This notebook provides an interactive **Guided Calibration Wizard** to extract the true physical acoustic constant $k(f_0)$ in $\text{V}\cdot\text{m}$ for a single carrier tone, validate the $1/r$ acoustic decay physics ($R^2 \ge 0.95$), and export a calibrated profile (`calibrated_room_profile.json`) for real-time metric distance tracking in Notebook 1.

---

### 🏛️ The Physical Acoustic Principle
Spherical direct-wave acoustic decay dictates:
$$V_{\text{RMS}}(r_i) = k(f_0) \cdot \left(\frac{1}{r_i}\right) + c_{\text{room}}$$

* **$k(f_0)$ [$\text{V}\cdot\text{m}$]:** The true physical source-to-microphone coupling constant at carrier pitch $f_0$.
* **$c_{\text{room}}$ [$\text{V}$]:** Ambient room reverberation and echo floor.
* **Dynamic Boundary Pruner:** Automatically excludes near-field speaker/ADC clipping at small $r$ and far-field echo noise floors at large $r$, fitting $k$ strictly on the direct-wave $1/r$ region.
* **Weighted Least Squares (WLS):** Weights each distance point by inverse statistical variance ($w_i = 1/\sigma_i^2$) across $N=30$ repeat frame observations.

## 1. Initialize Hardware Overlay & System Metadata
Load the FPGA overlay and record fixed traceability metadata (speaker device, volume level, and microphone gain position).

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    AcousticCalibrationProtocol,
    AcousticProfile,
    DistanceEstimator
)

# 1. Initialize FPGA Overlay
ol = MicrophoneArrayOverlay()

# 2. Traceability Metadata (Keep these physical parameters fixed during calibration)
system_metadata = {
    "speaker_device": "Smartphone",
    "speaker_volume_setting": 0.75,          # e.g. 75% volume slider
    "mic_gain_setting": "+35dB / 12 o clock", # Fixed MAX4466 potentiometer position
    "environment_label": "Acoustic_Lab_Desk",
    "temperature_c": 20.0
}

# 3. Initialize Calibration Protocol
protocol = AcousticCalibrationProtocol(r2_threshold=0.95, system_metadata=system_metadata)

print(f"✅ Hardware Overlay Active: {ol.current_profile} mode ({ol.fs_per_ch:.0f} SPS per channel)")
print(f"✅ System Metadata Locked: Volume={system_metadata['speaker_volume_setting']*100:.0f}%, Gain={system_metadata['mic_gain_setting']}")
print(f"✅ Quality Gate Initialized (Target R² >= {protocol.r2_threshold})")

## 2. Carrier Pitch Selection (Auto-Detect or Manual Fix)
Start playing your chosen tone on your phone/generator (e.g., $1000\,\text{Hz}$, $1500\,\text{Hz}$, or $2000\,\text{Hz}$). You can either let the software **auto-detect** the dominant tone or **manually fix** the exact frequency to track.

In [ ]:
# Set to True to sample the room and lock onto the loudest tone;
# Set to False to track an exact frequency manually specified.
AUTO_DETECT_DOMINANT = True
MANUAL_CARRIER_FREQ = 1500.0  # Used only if AUTO_DETECT_DOMINANT = False

if AUTO_DETECT_DOMINANT:
    print("🔍 Listening to room to auto-detect dominant carrier tone...")
    test_frame = ol.capture_quadruple(source="A0", f_min=100.0, f_max=15000.0, timeout=1.0)
    detected_pitch = test_frame["quadruple"]["frequency_hz"]
    detected_amp = test_frame["quadruple"]["amplitude_v"]
    
    if np.isfinite(detected_pitch) and detected_amp >= 0.003:
        carrier_freq = float(np.round(detected_pitch, 1))
        print(f"🎯 LOCKED onto dominant tone: {carrier_freq:.1f} Hz ({detected_amp*1000:.1f} mV RMS)")
    else:
        carrier_freq = MANUAL_CARRIER_FREQ
        print(f"⚠️ No active tone detected above noise floor. Falling back to manual: {carrier_freq:.1f} Hz")
else:
    carrier_freq = float(MANUAL_CARRIER_FREQ)
    print(f"🎯 Fixed target carrier frequency: {carrier_freq:.1f} Hz")

# Set search filter window around carrier (±50 Hz)
f_band_min = max(20.0, carrier_freq - 50.0)
f_band_max = carrier_freq + 50.0
print(f"   Narrow-band tracking filter: [{f_band_min:.1f} Hz -> {f_band_max:.1f} Hz]")

## 3. Guided Calibration Wizard (Single Interactive Cell)
Place a measuring tape along your desk. Run the cell below:
1. It prompts you to position the speaker at each distance in `calibration_distances_m`.
2. Move the speaker to the mark, then press **`[Enter]`**.
3. It automatically bursts $N=30$ frames ($0.00\,\mu\text{s}$ skew, BFP-immune coherent demodulation), prints the statistical confidence interval, and moves to the next station.
4. When the last station is reached, it automatically runs WLS regression, dynamic boundary pruning, and displays interactive diagnostic curves.

In [ ]:
import time

# Define your physical distance stations in meters (dense near-to-mid coverage)
calibration_distances_m = [
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00
]

protocol.clear()  # Clear any previous measurements
n_burst_samples = 30

print("=" * 75)
print(f"🚀 STARTING GUIDED CALIBRATION WIZARD (f0 = {carrier_freq:.1f} Hz, N = {n_burst_samples} samples/point)")
print(f"Stations to measure: {[f'{d*100:.0f}cm' for d in calibration_distances_m]}")
print("=" * 75)

# Interactive Guided Loop
for idx, dist_m in enumerate(calibration_distances_m):
    dist_cm = dist_m * 100.0
    # Wait for physical positioning
    prompt_msg = f"[{idx+1}/{len(calibration_distances_m)}] Place speaker at {dist_cm:5.1f} cm ({dist_m:.2f} m) and press [Enter]..."
    input(prompt_msg)
    
    # Capture N=30 burst observations
    samples = []
    for _ in range(n_burst_samples):
        frame = ol.capture_quadruple(source="A0", f_min=f_band_min, f_max=f_band_max, timeout=0.35)
        amp_v = frame["quadruple"]["amplitude_v"]
        if amp_v > 0.0005:
            samples.append(amp_v)
        time.sleep(0.005)
        
    if len(samples) < 5:
        print(f"    ⚠️ WARNING: Signal too weak or silence detected at {dist_cm:.1f} cm! Check speaker.")
        continue
        
    protocol.add_measurement(distance_m=dist_m, frequency_hz=carrier_freq, amplitude_v=samples)
    
    mean_v = float(np.mean(samples))
    std_v = float(np.std(samples, ddof=1)) if len(samples) > 1 else 0.0
    sem_v = std_v / np.sqrt(len(samples))
    print(f"    ✅ Captured: V_RMS = {mean_v*1000.0:6.2f} mV ± {sem_v*1000.0:4.2f} mV (σ={std_v*1000.0:4.2f} mV, N={len(samples)})")

print("\n" + "=" * 75)
print("📊 ALL STATIONS CAPTURED — EXECUTING WLS REGRESSION & DYNAMIC PRUNING...")
print("=" * 75)

# -----------------------------------------------------------------------------
# Execute WLS & Dynamic Power-Law Boundary Pruning
# -----------------------------------------------------------------------------
fit_results = protocol.fit()
res = fit_results[carrier_freq]

print(f"\n🏆 CALIBRATION FIT RESULTS (f0 = {carrier_freq:.1f} Hz):")
print(f"   • Acoustic Constant k(f0) : {res['k']:.4f} ± {res['delta_k']:.4f} V·m")
print(f"   • Room Reverberation c    : {res['c_room']*1000.0:.2f} mV")
print(f"   • Linearity Score R²      : {res['r_squared']:.4f} (Quality Gate >= {protocol.r2_threshold})")
print(f"   • Passed Quality Gate     : {'✅ YES' if res['passed_gate'] else '❌ NO'}")
print(f"   • Certified 1/r Window    : [{res['r_valid_min_m']*100:.0f} cm -> {res['r_valid_max_m']*100:.0f} cm] ({res['n_pruned_points']}/{res['n_total_points']} stations active)")

# -----------------------------------------------------------------------------
# Interactive Diagnostic Plots (Linear WLS vs Physical Hyperbolic Decay)
# -----------------------------------------------------------------------------
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"<b>1. Linear WLS 1/r Fit (f0 = {carrier_freq:.1f} Hz, R² = {res['r_squared']:.4f})</b>",
        f"<b>2. Physical Distance Decay Curve V_RMS(r)</b>"
    ),
    horizontal_spacing=0.10
)

stations = res["raw_stations"]
r_active = np.array([st["r_m"] for st in stations if st["is_pruned_in"]])
v_active = np.array([st["mean_v"] for st in stations if st["is_pruned_in"]]) * 1000.0
err_active = np.array([st["std_v"] for st in stations if st["is_pruned_in"]]) * 1000.0

r_pruned = np.array([st["r_m"] for st in stations if not st["is_pruned_in"]])
v_pruned = np.array([st["mean_v"] for st in stations if not st["is_pruned_in"]]) * 1000.0

# Panel 1: Linear 1/r Domain
if len(r_active) > 0:
    inv_r_active = 1.0 / r_active
    fig.add_scatter(
        x=inv_r_active, y=v_active,
        error_y=dict(type="data", array=err_active, visible=True),
        mode="markers", marker=dict(size=9, color="#00FFCC"),
        name="Active 1/r Points", row=1, col=1
    )
    inv_r_line = np.linspace(min(inv_r_active) * 0.9, max(inv_r_active) * 1.1, 50)
    v_line = (res["k"] * inv_r_line + res["c_room"]) * 1000.0
    fig.add_scatter(
        x=inv_r_line, y=v_line, mode="lines", line=dict(color="#00FFCC", dash="dash", width=2),
        name="WLS Fit Line", row=1, col=1
    )

if len(r_pruned) > 0:
    fig.add_scatter(
        x=1.0 / r_pruned, y=v_pruned, mode="markers",
        marker=dict(size=7, color="gray", symbol="x"),
        name="Pruned (Near/Echo Floor)", row=1, col=1
    )

# Panel 2: Physical Distance Domain [m]
r_smooth = np.linspace(min(r_active) * 0.85, max(r_active) * 1.15, 100)
v_phys_line = (res["k"] / r_smooth + res["c_room"]) * 1000.0

fig.add_scatter(
    x=r_smooth * 100.0, y=v_phys_line, mode="lines", line=dict(color="#FFA500", width=2),
    name="k/r Physical Decay", row=1, col=2
)
fig.add_scatter(
    x=r_active * 100.0, y=v_active,
    error_y=dict(type="data", array=err_active, visible=True),
    mode="markers", marker=dict(size=9, color="#00FFCC"),
    name="Measured V_RMS", row=1, col=2
)

fig.update_layout(template="plotly_dark", height=460, title="<b>Acoustic Calibration Laboratory: WLS Physical Modeling</b>")
fig.update_xaxes(title="1 / Distance [m⁻¹]", row=1, col=1)
fig.update_yaxes(title="In-Band V_RMS [mV]", row=1, col=1)
fig.update_xaxes(title="Physical Distance [cm]", row=1, col=2)
fig.update_yaxes(title="In-Band V_RMS [mV]", row=1, col=2)

fig.show()

## 4. Export Calibrated Profile to JSON
Save the certified profile artifact to `calibrated_room_profile.json`. This stores the evaluated $k(f_0)$, statistical uncertainties, metadata, and certified operational distance bounds $[r_{\text{min}}, r_{\text{max}}]$.

> **Note:** Notebook 1 (`01_realtime_kinematics_telemetry.ipynb`) and `KinematicsDashboard` will automatically detect and load this profile!

In [ ]:
export_path = "calibrated_room_profile.json"
saved_file = protocol.save_profile_json(
    filepath=export_path,
    name="Calibrated_SingleTone_Profile",
    description=f"WLS calibrated profile at {carrier_freq:.1f} Hz with dynamic boundary pruning and N=30 bursts"
)

print(f"🎉 SUCCESS: Saved calibrated profile to: {saved_file.resolve()}")
print("   Notebook 1 is now ready to track physical distances with certified accuracy!")

## 5. Live Single-Channel Distance Inversion Verification
Load the newly saved profile into `DistanceEstimator` and test inverted distance tracking in centimeters.

In [ ]:
# Load profile into runtime DistanceEstimator
loaded_profile = AcousticProfile.from_json(export_path)
estimator = DistanceEstimator(profile=loaded_profile, noise_gate_v=0.003)

input("\n📍 Move your speaker to an arbitrary distance (e.g. ~45 cm) and press [Enter] to test...")

frame = ol.capture_quadruple(source="A0", f_min=f_band_min, f_max=f_band_max, timeout=0.5)
result = estimator.process_frame(frame, source="A0")

print("\n--- Live Distance Inversion Result ---")
print(f"Detected Pitch f0  : {result['frequency_hz']:.1f} Hz")
print(f"In-Band Amplitude  : {result['amplitude_v']*1000.0:.2f} mV RMS")
print(f"Evaluated k(f0)    : {result['k_evaluated']:.4f} V·m")
print(f"Inverted Distance  : {result['distance_m']*100.0:.1f} cm (±{result['distance_err_m']*100.0:.1f} cm)")
print(f"Operational Status : {result['distance_status']}")

# Release hardware resources cleanly
ol.close()
print("\n🔒 Hardware resources cleanly released.")